# Reference benchmark for the EN→AR dubbing translation step (WMT24++, speech domain)

Public data only: WMT24++ (`google/wmt24pp`, Apache-2.0), English → Arabic, domain `speech`.
Reference = `original_target` (MSA human translation; `target` is a dialect post-edit).
Code: https://github.com/HamzaAlSamman/AI-Dubbing-Project (branch below), `translation/dubbing_translation/benchmark.py`.
Every result is written to `/kaggle/working/evaluations/` with the git commit, config hash and test-set hash.

In [ ]:
import os, subprocess, sys, json, time
BRANCH = os.environ.get('BENCH_BRANCH', 'evaluation-v2')
REPO = '/kaggle/working/repo'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'bitsandbytes',
                'sacrebleu', 'datasets', 'sentencepiece', 'google-genai', 'num2words'], check=True)
sys.path.insert(0, REPO + '/translation')
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
import torch, transformers
print('torch', torch.__version__, 'transformers', transformers.__version__, 'gpus', torch.cuda.device_count())

In [ ]:
# optional secrets (Kaggle: Add-ons -> Secrets). Without GEMINI_API_KEY the Gemini system is skipped.
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for k in ('GEMINI_API_KEY', 'HF_TOKEN'):
        try:
            os.environ[k] = _s.get_secret(k)
        except Exception:
            pass
except ImportError:
    pass
print({k: bool(os.environ.get(k)) for k in ('GEMINI_API_KEY', 'HF_TOKEN')})

In [ ]:
from pathlib import Path
from dubbing_translation.io import load_config
from dubbing_translation.benchmark import as_tasks, generate_cached, load_wmt24pp_hf, run_reference

CFG = load_config()
RATE = CFG['voice']['letters_per_second']
WORK = Path('/kaggle/working')
items = load_wmt24pp_hf('en-ar_SA', domains=('speech',))
LIMIT = int(os.environ.get('BENCH_LIMIT', '0'))
if LIMIT:
    items = items[:LIMIT]
btasks = as_tasks(items)
print(len(items), 'segments;', sum(len(i['source_text'].split()) for i in items), 'source words')

In [ ]:
from dubbing_translation.generators import HFChatMT, HyMT, IsometricLLM, LLMClient

m = CFG['models']
GENERATORS = {
    'hy_mt15_7b':        lambda: HyMT(m['mt']),
    'hy_mt2_7b':         lambda: HFChatMT('tencent/Hy-MT2-7B', 'hy_mt2', samples=0),
    'qwen3_4b_baseline': lambda: HFChatMT('Qwen/Qwen3-4B', 'msa_plain', samples=0,
                                          template_kwargs={'enable_thinking': False}),
    'jais3_7b':          lambda: HFChatMT('inception42/Jais-3-7B', 'msa_plain', samples=0, trust_remote_code=True),
}
if os.environ.get('GEMINI_API_KEY'):
    GENERATORS['gemini_isometric'] = lambda: IsometricLLM(LLMClient(m['llm_provider'], m['llm_model'],
                                                                    min_interval_s=m['llm_min_interval_s'],
                                                                    timeout_s=m.get('llm_timeout_s', 180)))
systems, prov, timing, failures = {}, {}, {}, {}
for name, make in GENERATORS.items():
    t0, gen = time.time(), None
    try:
        gen = make()
        systems[name] = generate_cached(gen, btasks, WORK / 'bench_outputs', rate=RATE, budget_cfg=CFG['budget'])
        prov[name] = gen.name
    except Exception as exc:
        failures[name] = f'{type(exc).__name__}: {exc}'
        print('FAILED', name, failures[name])
    finally:
        if gen is not None and hasattr(gen, 'close'):
            try:
                gen.close()
            except Exception:
                pass
    timing[name] = round(time.time() - t0, 1)
    print(name, len(systems.get(name, {})), 'outputs', timing[name], 's', flush=True)

In [ ]:
from dubbing_translation.evaluate import Evaluation
from dubbing_translation.scoring import MetricXRef

ref_metric = MetricXRef(m['qe'])
ev = Evaluation(WORK, items, CFG, label='wmt24pp_speech', repo_dir=REPO, qe_name=ref_metric.name)
summary = run_reference(ev, items, systems, rate=RATE, ref_scorer=ref_metric, provenance=prov,
                        test_set='wmt24pp_en-ar_SA_speech_original_target')
ref_metric.close()
json.dump({'summary': summary, 'generation_seconds': timing, 'failures': failures, 'eval_id': ev.eval_id,
           'n_items': len(items)}, open(WORK / 'benchmark_summary.json', 'w'), ensure_ascii=False, indent=1)
cols = ['units', 'chrf_pp', 'bleu', 'metricx_ref_mean', 'len_ratio_ref_median', 'lc10_pct',
        'fits_or_speedup_pct', 'too_long_pct', 'too_short_pct']
lines = ['| system | ' + ' | '.join(cols) + ' |', '|' + '---|' * (len(cols) + 1)]
for name, s in summary.items():
    lines.append(f'| {name} | ' + ' | '.join(str(s.get(c, '')) for c in cols) + ' |')
open(WORK / 'benchmark_summary.md', 'w').write('
'.join(lines) + '
')
print('
'.join(lines)); print('failures:', failures)